# Paper — 01b: Orientation Estimator Comparison

Isolates where the 0°/45°/90°/135° spike artifact originates by comparing four orientation estimators on the same detections.

**Methods tested:**
1. Current pipeline: `segmentize → EllipseModel → MRR` (baseline)
2. Ellipse θ direct: `segmentize → EllipseModel → θ` (skip MRR)
3. No segmentize: `raw contour → EllipseModel → MRR`
4. Polygon moments: Green's theorem area PCA (no ellipse fit, no segmentize)

**Experiments:**
- **A.** Method × size bin grid (main diagnostic)
- **B.** Segmentize density ablation
- **C.** Cross-model comparison (YOLO, SAM2 zero-shot, SAM2 fine-tuned, SAM2-auto)

**Predictions:**
- Methods 1 ≈ 2 → MRR is not the source of bias
- Method 3 differs from 1 → segmentize amplifies bias
- Method 4 is flatter → contour-based ellipse fit is the culprit
- Spikes weaken for larger size bins → rasterization artifact

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
from shapely import segmentize
from scipy.stats import kstest
from tqdm import tqdm

from rastertools_BOULDERING import metadata as raster_metadata
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

In [ ]:
work_dir  = Path.home() / "tmp" / "YOLOv8BeyondEarth"
in_raster = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")

PRED_CONFIGS = {
    "YOLOv8":          (work_dir / "exp_yolo_256",           "*-downscaled-mask-nms.shp"),
    "SAM2 zero-shot":  (work_dir / "exp_sam2_256",           "*-downscaled-mask-nms.shp"),
    "SAM2 fine-tuned": (work_dir / "exp_sam2_finetuned_256", "*-downscaled-mask-nms.shp"),
    "SAM2-auto":       (work_dir / "exp_sam2_auto_256",      "*-mask-nms.shp"),
}

res             = raster_metadata.get_resolution(in_raster)[0]
AREAL_THRESHOLD = (res ** 2) * (4.74 ** 2)
AR_MIN, AR_MAX  = 1.2, 2.0
BINS            = np.linspace(0, 180, 37)

# Set to an integer (e.g. 5000) to subsample for quick testing, None for full run
MAX_N = None

OUT_DIR = Path("figures_paper"); OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.titlesize": 9, "figure.dpi": 150})
print(f"Resolution: {res:.4f} m/px   Areal threshold: {AREAL_THRESHOLD:.4f} m²")

In [ ]:
def _ellipse_mrr(poly, seg_res):
    """Shared core: optionally segmentize → ellipse fit → MRR → (angle180, AR)."""
    try:
        geom = segmentize(poly, seg_res) if seg_res is not None else poly
        row = pd.Series({"geometry": geom})
        ellipse_poly, _, _, _ = fitEllipse(row)
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        _, _, long_ax, short_ax, _, _, _, angle180 = boulder_row(mrr_row)
        if short_ax < 1e-6:
            return None
        return angle180, long_ax / short_ax
    except Exception:
        return None


def m1_current(poly):
    """Method 1 (baseline): segmentize(res) → EllipseModel → MRR."""
    return _ellipse_mrr(poly, res)


def m2_theta(poly):
    """Method 2: segmentize(res) → EllipseModel → theta directly (skip MRR)."""
    try:
        row_seg = pd.Series({"geometry": segmentize(poly, res)})
        _, a, b, theta = fitEllipse(row_seg)
        if b < 1e-6:
            return None
        # EllipseModel theta: angle from x-axis (East), CCW
        # convert to azimuth from North: (90 - degrees(theta)) % 180
        angle180 = (90.0 - np.degrees(theta)) % 180
        return angle180, a / b
    except Exception:
        return None


def m3_no_seg(poly):
    """Method 3: raw contour (no segmentize) → EllipseModel → MRR."""
    return _ellipse_mrr(poly, None)


def m4_moments(poly):
    """Method 4: polygon area moments via Green's theorem (no ellipse fit, no segmentize)."""
    try:
        coords = np.array(poly.exterior.coords[:-1])
        cx, cy = poly.centroid.x, poly.centroid.y
        x, y   = coords[:, 0] - cx, coords[:, 1] - cy
        xn, yn = np.roll(x, -1), np.roll(y, -1)
        cross  = x * yn - xn * y
        A = 0.5 * np.abs(cross.sum())
        if A < 1e-12:
            return None
        mu20 = np.sum((x**2 + x*xn + xn**2) * cross) / (6 * A)
        mu02 = np.sum((y**2 + y*yn + yn**2) * cross) / (6 * A)
        mu11 = np.sum((x*yn + 2*x*y + 2*xn*yn + xn*y) * cross) / (24 * A)
        cov  = np.array([[mu20, mu11], [mu11, mu02]])
        eigvals, eigvecs = np.linalg.eigh(cov)   # ascending order
        long_vec = eigvecs[:, 1]                 # largest eigenvalue
        angle180 = np.degrees(np.arctan2(long_vec[0], long_vec[1])) % 180
        ar = np.sqrt(np.abs(eigvals[1]) / max(np.abs(eigvals[0]), 1e-12))
        return angle180, max(ar, 1 / ar if ar > 0 else 1.0)
    except Exception:
        return None


METHODS = {
    "1 · segmentize→ellipse→MRR": m1_current,
    "2 · ellipse θ direct":       m2_theta,
    "3 · no segmentize→ellipse": m3_no_seg,
    "4 · moments (area PCA)":     m4_moments,
}
print("Estimators defined:", list(METHODS.keys()))

In [ ]:
pred_dir, glob_pat = PRED_CONFIGS["YOLOv8"]
shp_paths = sorted(pred_dir.glob(glob_pat))
assert shp_paths, f"No shapefiles found in {pred_dir}"

gdfs = [gpd.read_file(p) for p in shp_paths]
gdf  = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
gdf["poly_area"] = gdf.geometry.area
gdf = gdf[gdf["poly_area"] >= AREAL_THRESHOLD].reset_index(drop=True)
gdf["d_px"] = np.sqrt(4 * gdf["poly_area"] / np.pi) / res

if MAX_N is not None and len(gdf) > MAX_N:
    gdf = gdf.sample(MAX_N, random_state=42).reset_index(drop=True)
    print(f"Subsampled to {MAX_N:,} detections")

print(f"YOLO detections after area filter: {len(gdf):,}")
print(f"\nEquivalent pixel diameter:")
print(gdf["d_px"].describe().round(2))

In [ ]:
results = {name: [] for name in METHODS}

for geom in tqdm(gdf.geometry, desc="Running all methods"):
    if geom is None or geom.is_empty:
        for name in METHODS:
            results[name].append(None)
        continue
    for name, fn in METHODS.items():
        results[name].append(fn(geom))

for name in METHODS:
    angles_all = [r[0] if r else np.nan for r in results[name]]
    ar_all     = [r[1] if r else np.nan for r in results[name]]
    gdf[name + "_angle"] = angles_all
    gdf[name + "_ar"]    = ar_all
    # apply AR filter
    bad = ~((gdf[name + "_ar"] >= AR_MIN) & (gdf[name + "_ar"] <= AR_MAX))
    gdf.loc[bad, name + "_angle"] = np.nan
    n = gdf[name + "_angle"].notna().sum()
    print(f"{name:<40}  {n:>7,} elongated boulders")

## Experiment A — Method × size bin grid

If spikes weaken from small → large: rasterization artifact confirmed.
If methods 1 ≈ 2: MRR is not the source.
If method 3 differs from 1: segmentize is amplifying bias.
If method 4 is flatter: contour-based ellipse fit is the culprit.

In [ ]:
SIZE_BINS = [
    (0,    8,   "small  (< 8 px)"),
    (8,    20,  "medium (8–20 px)"),
    (20,   1e9, "large  (> 20 px)"),
]

n_methods = len(METHODS)
n_bins    = len(SIZE_BINS)
fig, axes = plt.subplots(n_bins, n_methods,
                         figsize=(3.2 * n_methods, 2.6 * n_bins),
                         sharex=True)

for row_i, (dlo, dhi, size_label) in enumerate(SIZE_BINS):
    size_mask = (gdf["d_px"] >= dlo) & (gdf["d_px"] < dhi)
    for col_j, name in enumerate(METHODS):
        ax = axes[row_i][col_j]
        angles = gdf.loc[size_mask, name + "_angle"].dropna().values
        counts, _ = np.histogram(angles, bins=BINS)
        cx = (BINS[:-1] + BINS[1:]) / 2
        D, p = (kstest(angles / 180.0, "uniform") if len(angles) >= 10
                else (np.nan, np.nan))
        ax.bar(cx, counts, width=4.5, color="#4C72B0", edgecolor="white", lw=0.3)
        ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.6, alpha=0.5)
        ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180])
        ax.set_title(f"{name}\n{size_label}  n={len(angles):,}  D={D:.3f}", fontsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        if col_j == 0:
            ax.set_ylabel("Count")
        if row_i == n_bins - 1:
            ax.set_xlabel("Orientation (°)")

fig.suptitle("YOLO — Orientation method × boulder size", y=1.01, fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_method_size_grid.pdf", bbox_inches="tight")
plt.show()
print("Saved fig_method_size_grid.pdf")

## KS statistics — uniformity test

D-statistic: distance from uniform. Higher D = more biased.
p-value: probability of seeing this D under a uniform distribution.

In [ ]:
rows = []
for dlo, dhi, size_label in SIZE_BINS:
    size_mask = (gdf["d_px"] >= dlo) & (gdf["d_px"] < dhi)
    for name in METHODS:
        angles = gdf.loc[size_mask, name + "_angle"].dropna().values
        if len(angles) >= 10:
            D, p = kstest(angles / 180.0, "uniform")
        else:
            D, p = np.nan, np.nan
        rows.append({"Method": name, "Size bin": size_label,
                     "n": len(angles), "D": D, "p": p})

ks_df = pd.DataFrame(rows)
ks_df["D"] = ks_df["D"].round(3)
ks_df["p"] = ks_df["p"].apply(lambda x: f"{x:.2e}" if not np.isnan(x) else "—")
print(ks_df.to_string(index=False))

## Experiment B — Segmentize density ablation

Varies segmentize resolution to test whether densifying the contour amplifies the orientation bias.
All variants use the same ellipse→MRR pipeline (method 1).

In [ ]:
SEG_DENSITIES = {
    "no segmentize":   None,
    "coarse (4× res)": res * 4,
    "current (1× res)": res,
    "fine (0.25× res)": res / 4,
}

seg_results = {name: [] for name in SEG_DENSITIES}

for geom in tqdm(gdf.geometry, desc="Segmentize ablation"):
    if geom is None or geom.is_empty:
        for name in SEG_DENSITIES:
            seg_results[name].append(None)
        continue
    for seg_name, seg_res in SEG_DENSITIES.items():
        r = _ellipse_mrr(geom, seg_res)
        seg_results[seg_name].append(
            r[0] if (r and AR_MIN <= r[1] <= AR_MAX) else None)

fig, axes = plt.subplots(1, len(SEG_DENSITIES),
                         figsize=(3.0 * len(SEG_DENSITIES), 2.8),
                         sharex=True)
for ax, (seg_name, angle_list) in zip(axes, seg_results.items()):
    angles = np.array([a for a in angle_list if a is not None])
    counts, _ = np.histogram(angles, bins=BINS)
    cx = (BINS[:-1] + BINS[1:]) / 2
    D, p = (kstest(angles / 180.0, "uniform") if len(angles) >= 10
            else (np.nan, np.nan))
    ax.bar(cx, counts, width=4.5, color="#C44E52", edgecolor="white", lw=0.3)
    ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.6, alpha=0.5)
    ax.set_title(f"{seg_name}\nn={len(angles):,}  D={D:.3f}", fontsize=8)
    ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180], xlabel="Orientation (°)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Count")

fig.suptitle("YOLO — Segmentize density ablation (ellipse→MRR)", y=1.02, fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_segmentize_ablation.pdf", bbox_inches="tight")
plt.show()
print("Saved fig_segmentize_ablation.pdf")

## Experiment C — Cross-model comparison

Runs all four estimators across YOLO, SAM2 zero-shot, SAM2 fine-tuned, and SAM2-auto.
Shows whether the artifact pattern differs across detection methods.

In [ ]:
model_method_angles = {}  # model_name → method_name → angle array

for model_name, (pred_dir_m, glob_m) in PRED_CONFIGS.items():
    shps = sorted(pred_dir_m.glob(glob_m))
    if not shps:
        print(f"  [{model_name}] no shapefiles — skipping")
        continue
    print(f"\nLoading {model_name}...")
    gdfs_m = [gpd.read_file(p) for p in shps]
    gdf_m  = gpd.GeoDataFrame(pd.concat(gdfs_m, ignore_index=True), crs=gdfs_m[0].crs)
    gdf_m["poly_area"] = gdf_m.geometry.area
    gdf_m = gdf_m[gdf_m["poly_area"] >= AREAL_THRESHOLD].reset_index(drop=True)
    if MAX_N is not None and len(gdf_m) > MAX_N:
        gdf_m = gdf_m.sample(MAX_N, random_state=42).reset_index(drop=True)

    model_method_angles[model_name] = {}
    for mname, mfn in METHODS.items():
        angles = []
        for geom in tqdm(gdf_m.geometry, desc=f"  {mname[:20]}", leave=False):
            if geom is None or geom.is_empty:
                continue
            r = mfn(geom)
            if r and AR_MIN <= r[1] <= AR_MAX:
                angles.append(r[0])
        model_method_angles[model_name][mname] = np.array(angles)
        print(f"    {mname[:40]:<40}  {len(angles):>7,} elongated")

n_models  = len(model_method_angles)
n_methods = len(METHODS)
fig, axes = plt.subplots(n_models, n_methods,
                         figsize=(3.0 * n_methods, 2.5 * n_models),
                         sharex=True)

for row_i, (model_name, method_dict) in enumerate(model_method_angles.items()):
    for col_j, mname in enumerate(METHODS):
        ax = axes[row_i][col_j]
        angles = method_dict.get(mname, np.array([]))
        counts, _ = np.histogram(angles, bins=BINS)
        cx = (BINS[:-1] + BINS[1:]) / 2
        D, p = (kstest(angles / 180.0, "uniform") if len(angles) >= 10
                else (np.nan, np.nan))
        ax.bar(cx, counts, width=4.5, color="#4C72B0", edgecolor="white", lw=0.3)
        ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.6, alpha=0.5)
        ax.set_title(f"{model_name}\n{mname}\nn={len(angles):,}  D={D:.3f}", fontsize=6)
        ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180])
        ax.spines[["top", "right"]].set_visible(False)
        if col_j == 0:
            ax.set_ylabel("Count")
        if row_i == n_models - 1:
            ax.set_xlabel("Orientation (°)")

fig.suptitle("All models × all methods", y=1.01, fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_model_method_grid.pdf", bbox_inches="tight")
plt.show()
print("Saved fig_model_method_grid.pdf")

## Proposed Fixes

Two approaches that avoid the staircase contour entirely:

**Fix A — Smooth contour (σ=1.0 and σ=2.0 px)**
Rasterize the polygon at native resolution → Gaussian blur → re-threshold at 0.5 → extract contour → ellipse → MRR.
Rounds off staircase corners before polygon extraction. No raster needed.

**Fix B — Structure tensor on image gradients**
Compute Sobel gradients on the original image inside the mask → build structure tensor → long axis = eigenvector of *smallest* eigenvalue (perpendicular to dominant gradient direction). Completely avoids binary mask geometry. Requires the raster.

Compared against baseline (Method 1). AR filter for fixes A uses the MRR AR from the smooth polygon. For fix B, AR is computed from Method 4 (moments) to avoid using a gradient-based proxy as a filter.

In [ ]:
import cv2
import rasterio
from rasterio.mask import mask as rio_mask
from scipy.ndimage import gaussian_filter as nd_gaussian, sobel as nd_sobel


# ── Fix A: Smooth contour ─────────────────────────────────────────────────

def m_smooth_contour(poly, sigma):
    """Rasterize poly at native res → Gaussian blur (sigma px) → contour → ellipse → MRR."""
    try:
        ppu = 1.0 / res                              # pixels per geographic unit
        minx, miny, maxx, maxy = poly.bounds
        pad = max(5, int(sigma * 3))
        w = int((maxx - minx) * ppu) + 2 * pad
        h = int((maxy - miny) * ppu) + 2 * pad
        if w < 6 or h < 6:
            return None
        pts    = np.array(poly.exterior.coords[:-1])
        pts_px = ((pts - [minx, miny]) * ppu + pad).astype(np.int32)
        canvas = np.zeros((h, w), np.float32)
        cv2.fillPoly(canvas, [pts_px], 1.0)
        blurred   = nd_gaussian(canvas, sigma=sigma)
        smooth_u8 = (blurred > 0.5).astype(np.uint8) * 255
        cnts, _ = cv2.findContours(smooth_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not cnts:
            return None
        cnt = max(cnts, key=cv2.contourArea).squeeze()
        if cnt.ndim != 2 or len(cnt) < 4:
            return None
        coords_geo = (cnt.astype(float) - pad) / ppu + [minx, miny]
        from shapely.geometry import Polygon as SPoly
        smooth_poly = SPoly(coords_geo)
        if not smooth_poly.is_valid or smooth_poly.area < 1e-12:
            return None
        return _ellipse_mrr(smooth_poly, None)      # already smooth — no segmentize
    except Exception:
        return None


def m5_smooth_sigma1(poly):
    """Fix A1: smooth contour σ=1.0 px."""
    return m_smooth_contour(poly, sigma=1.0)


def m6_smooth_sigma2(poly):
    """Fix A2: smooth contour σ=2.0 px."""
    return m_smooth_contour(poly, sigma=2.0)


# ── Fix B: Structure tensor ───────────────────────────────────────────────

def m7_structure_tensor(poly):
    """Fix B: structure tensor from image gradients inside mask.
    Long axis = eigenvector of smallest eigenvalue of J (perpendicular to dominant gradient).
    Returns (angle180, None) — AR is not meaningful from the tensor; filter via moments AR.
    """
    try:
        with rasterio.open(in_raster) as src:
            out_image, out_transform = rio_mask(src, [poly], crop=True, nodata=0)
            img = out_image[0].astype(float)

        interior = img != 0
        if interior.sum() < 9:
            return None

        # Sobel in image space: axis=1 → col direction (East), axis=0 → row direction (South)
        Gx_img = nd_sobel(img, axis=1)   # East gradient
        Gy_img = nd_sobel(img, axis=0)   # South gradient

        # Structure tensor in (East, South) image coordinates
        J_ee = np.sum(Gx_img[interior] ** 2)
        J_ss = np.sum(Gy_img[interior] ** 2)
        J_es = np.sum(Gx_img[interior] * Gy_img[interior])

        if J_ee + J_ss < 1e-12:
            return None

        J = np.array([[J_ee, J_es],
                      [J_es, J_ss]])
        eigvals, eigvecs = np.linalg.eigh(J)   # ascending order

        # Smallest eigenvalue eigenvector = long axis direction in (East, South) image coords
        ev_east  =  eigvecs[0, 0]    # East component
        ev_south =  eigvecs[1, 0]    # South component
        # Convert to geographic azimuth: arctan2(East, North) where North = -South
        angle180 = np.degrees(np.arctan2(ev_east, -ev_south)) % 180

        # Return a dummy AR of 0 so the caller can apply moments-based AR filter separately
        return angle180, 0.0
    except Exception:
        return None


FIX_METHODS = {
    "baseline (method 1)":     m1_current,
    "fix A1: smooth σ=1.0px":  m5_smooth_sigma1,
    "fix A2: smooth σ=2.0px":  m6_smooth_sigma2,
    "fix B: structure tensor":  m7_structure_tensor,
}
print("Proposed fix estimators defined:", list(FIX_METHODS.keys()))

In [ ]:
# Pre-compute moments AR for use as the filter for Fix B (structure tensor)
moments_ar_col = "4 · moments (area PCA)_ar"

fix_angles = {name: [] for name in FIX_METHODS}

for i, (geom, m4_ar) in enumerate(tqdm(
        zip(gdf.geometry, gdf[moments_ar_col]),
        total=len(gdf), desc="Running proposed fixes")):

    if geom is None or geom.is_empty:
        for name in FIX_METHODS:
            fix_angles[name].append(np.nan)
        continue

    for name, fn in FIX_METHODS.items():
        r = fn(geom)
        if r is None:
            fix_angles[name].append(np.nan)
            continue

        angle, ar = r

        if name == "fix B: structure tensor":
            # Use moments AR for filter (tensor AR proxy is not geometric)
            passes = (not np.isnan(m4_ar)) and (AR_MIN <= m4_ar <= AR_MAX)
        else:
            passes = (AR_MIN <= ar <= AR_MAX)

        fix_angles[name].append(angle if passes else np.nan)

for name in FIX_METHODS:
    gdf[name + "_fx"] = fix_angles[name]
    n = pd.Series(fix_angles[name]).notna().sum()
    print(f"{name:<40}  {n:>7,} elongated boulders")

# ── Figure: proposed fixes × size bins ───────────────────────────────────
n_fix = len(FIX_METHODS)
fig, axes = plt.subplots(len(SIZE_BINS), n_fix,
                         figsize=(3.2 * n_fix, 2.6 * len(SIZE_BINS)),
                         sharex=True)

COLORS = ["#4C72B0", "#DD8452", "#C44E52", "#55A868"]

for row_i, (dlo, dhi, size_label) in enumerate(SIZE_BINS):
    size_mask = (gdf["d_px"] >= dlo) & (gdf["d_px"] < dhi)
    for col_j, name in enumerate(FIX_METHODS):
        ax = axes[row_i][col_j]
        angles = gdf.loc[size_mask, name + "_fx"].dropna().values
        counts, _ = np.histogram(angles, bins=BINS)
        cx = (BINS[:-1] + BINS[1:]) / 2
        D, p = (kstest(angles / 180.0, "uniform") if len(angles) >= 10
                else (np.nan, np.nan))
        ax.bar(cx, counts, width=4.5, color=COLORS[col_j], edgecolor="white", lw=0.3)
        ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.6, alpha=0.5)
        ax.set(xlim=(0, 180), xticks=[0, 45, 90, 135, 180])
        ax.set_title(f"{name}\n{size_label}  n={len(angles):,}  D={D:.3f}", fontsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        if col_j == 0:
            ax.set_ylabel("Count")
        if row_i == len(SIZE_BINS) - 1:
            ax.set_xlabel("Orientation (°)")

fig.suptitle("YOLO — Proposed fixes vs baseline", y=1.01, fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_proposed_fixes.pdf", bbox_inches="tight")
plt.show()
print("Saved fig_proposed_fixes.pdf")

# ── KS summary ───────────────────────────────────────────────────────────
print(f"\n{'Method':<40}  {'Size bin':<22}  {'n':>6}  D-stat")
print("-" * 80)
for dlo, dhi, size_label in SIZE_BINS:
    size_mask = (gdf["d_px"] >= dlo) & (gdf["d_px"] < dhi)
    for name in FIX_METHODS:
        angles = gdf.loc[size_mask, name + "_fx"].dropna().values
        D = kstest(angles / 180.0, "uniform").statistic if len(angles) >= 10 else np.nan
        print(f"{name:<40}  {size_label:<22}  {len(angles):>6}  {D:.3f}" if not np.isnan(D)
              else f"{name:<40}  {size_label:<22}  {len(angles):>6}  —")
    print()